# Create the is_fraud column

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import sys, os, glob
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, DateType, DoubleType, FloatType
from datetime import datetime, timedelta

In [2]:
sys.path.insert(0, "../scripts")
from spark_setup import get_spark
spark = get_spark()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/20 15:16:58 WARN Utils: Your hostname, CompuPau, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/09/20 15:16:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/pau_l/project-2/venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/20 15:16:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# bring datasets
merchant_fraud = pd.read_parquet("../data/curated/merchant_fraud_labels.parquet")
consumer_fraud = pd.read_parquet("../data/curated/consumer_fraud_labels.parquet")
transactions = spark.read.parquet("../data/curated/df_transactions")

In [4]:
transactions.count() 

13614675

In [5]:
def nan_a_null(df):
    """Convert NaN columns to NULL values."""
    numeric_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, (DoubleType, FloatType))]
    for c in numeric_cols:
        df = df.withColumn(c, F.when(F.isnan(F.col(c)), None).otherwise(F.col(c)))
    return df

# to Spark
merchant_fraud_spark = nan_a_null(spark.createDataFrame(merchant_fraud))
consumer_fraud_spark = nan_a_null(spark.createDataFrame(consumer_fraud))

/home/pau_l/project-2/venv/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/pau_l/project-2/venv/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/pau_l/project-2/venv/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/pau_l/project-2/venv/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:936: FutureWarning: PySpark does not yet fully support pandas 

In [6]:
# join transaction with is_fraud column
transactions = transactions.join(merchant_fraud_spark, on="merchant_abn", how="left")
transactions = transactions.join(consumer_fraud_spark, on="order_id", how="left") 

In [7]:
transactions.count() 

26/09/20 15:19:28 WARN TaskSetManager: Stage 4 contains a task of very large size (196216 KiB). The maximum recommended task size is 1000 KiB.


13614675

In [8]:
# if one of the is_fraud -> 1
# if both of them NaN -> NaN
# else 0
transactions = transactions.withColumn(
    "is_fraud",
    F.when(
        F.col("is_fraud_merchant").isNull() & F.col("is_fraud_consumer").isNull(),
        F.lit(None).cast(DoubleType())
    )
    .when(
        (F.col("is_fraud_merchant") == 1.0) | (F.col("is_fraud_consumer") == 1.0),
        F.lit(1.0)
    )
    .otherwise(F.lit(0.0))
)

print(f"Total rows: {transactions.count():,}")
transactions.groupBy("is_fraud").count().show()

26/09/20 15:19:44 WARN TaskSetManager: Stage 14 contains a task of very large size (196216 KiB). The maximum recommended task size is 1000 KiB.


Total rows: 13,614,675


26/09/20 15:20:04 WARN TaskSetManager: Stage 24 contains a task of very large size (196216 KiB). The maximum recommended task size is 1000 KiB.


+--------+-------+
|is_fraud|  count|
+--------+-------+
|     1.0| 509569|
|    NULL|7973507|
|     0.0|5131599|
+--------+-------+



In [9]:
n_total = transactions.count()

summary = transactions.groupBy("is_fraud").count().withColumn(
    "pct", F.round(F.col("count") / n_total * 100, 2)
).orderBy(F.col("is_fraud").isNull().desc(), "is_fraud")

summary.show()

26/09/20 15:20:23 WARN TaskSetManager: Stage 34 contains a task of very large size (196216 KiB). The maximum recommended task size is 1000 KiB.
26/09/20 15:20:31 WARN TaskSetManager: Stage 44 contains a task of very large size (196216 KiB). The maximum recommended task size is 1000 KiB.


+--------+-------+-----+
|is_fraud|  count|  pct|
+--------+-------+-----+
|    NULL|7973507|58.57|
|     0.0|5131599|37.69|
|     1.0| 509569| 3.74|
+--------+-------+-----+



In [10]:
# compare to the fraud probability records before
n_null_fraud_prob = transactions.filter(F.col("fraud_probability").isNull()).count()
n_null_avg_merchant = transactions.filter(F.col("avg_merchant_fraud_prob").isNull()).count()

print(f"fraud_probability        - NULL: {n_null_fraud_prob:,} ({n_null_fraud_prob/n_total:.1%})")
print(f"avg_merchant_fraud_prob  - NULL: {n_null_avg_merchant:,} ({n_null_avg_merchant/n_total:.1%})")

26/09/20 15:20:51 WARN TaskSetManager: Stage 54 contains a task of very large size (196216 KiB). The maximum recommended task size is 1000 KiB.
26/09/20 15:21:00 WARN TaskSetManager: Stage 64 contains a task of very large size (196216 KiB). The maximum recommended task size is 1000 KiB.


fraud_probability        - NULL: 13,543,038 (99.5%)
avg_merchant_fraud_prob  - NULL: 13,031,129 (95.7%)


Comparing the results with the fraud probability records that we had before, we were able to reduce the Null values from 99% to 58.57%. Therefore, even though we still have records where it wasn't possible to determine whether there was fraud or not, the models performed for merchants and consumers allowed us to at least identify more fraudulent transactions. The ones that are still Null values cannot be determined with high certainty.

In [11]:
# complete version
transactions.write.mode("overwrite").parquet("../data/curated/transactions_with_is_fraud_full")

26/09/20 15:21:21 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/09/20 15:21:21 WARN TaskSetManager: Stage 75 contains a task of very large size (196216 KiB). The maximum recommended task size is 1000 KiB.


In [12]:
transactions_final = transactions.drop(
    "merchant_fraud_score_final",
    "is_fraud_merchant",
    "consumer_fraud_score_final",
    "is_fraud_consumer"
)

transactions_final.printSchema() 

root
 |-- order_id: string (nullable = true)
 |-- merchant_abn: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- order_datetime: date (nullable = true)
 |-- dollar_value: double (nullable = true)
 |-- consumer_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postcode: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- fraud_probability: double (nullable = true)
 |-- is_same_day_duplicate: boolean (nullable = true)
 |-- merchant_name: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- category: string (nullable = true)
 |-- revenue_band: string (nullable = true)
 |-- take_rate: double (nullable = true)
 |-- category_group: string (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- n_transactions: long (nullable = true)
 |-- avg_order_value: double (nullable = true)
 |-- avg_merchant_fraud_prob: double (nullable = true)
 |-- i

In [13]:
transactions_final.show(5)

26/09/20 15:22:10 WARN TaskSetManager: Stage 80 contains a task of very large size (196216 KiB). The maximum recommended task size is 1000 KiB.


+--------------------+------------+-------+--------------+------------------+-----------+-----------------+--------------------+-----+--------+------+------------------+---------------------+--------------------+--------------------+--------------------+------------+---------+-------------------+------------------+--------------+------------------+-----------------------+--------+
|            order_id|merchant_abn|user_id|order_datetime|      dollar_value|consumer_id|             name|             address|state|postcode|gender| fraud_probability|is_same_day_duplicate|       merchant_name|                tags|            category|revenue_band|take_rate|     category_group|     total_revenue|n_transactions|   avg_order_value|avg_merchant_fraud_prob|is_fraud|
+--------------------+------------+-------+--------------+------------------+-----------+-----------------+--------------------+-----+--------+------+------------------+---------------------+--------------------+--------------------

In [14]:
# only keep is_fraud column
transactions_final.write.mode("overwrite").parquet("../data/curated/transactions_with_is_fraud")

26/09/20 15:22:21 WARN TaskSetManager: Stage 89 contains a task of very large size (196216 KiB). The maximum recommended task size is 1000 KiB.
